# 03_gold_aggregations — Analytical Gold Tables
## ClinVar Medallion Pipeline · Gold Layer

Builds four analytical Gold tables from the clean Silver data.

**What happens here:**
- Read the Silver table, keeping clean rows only (`dq_flag = False`)
- Build four purpose-built aggregation tables, each answering one question
- Export the tables to CSV for the dashboard
- DLT-ready: the managed-pipeline structure is shown in comments

**Gold tables produced:**
- `significance_summary`    — variant counts by clinical significance
- `pathogenic_by_gene`      — top genes by actionable variant count
- `variants_by_chromosome`  — variant distribution and pathogenic proportion per chromosome
- `condition_burden`        — top conditions by actionable variant burden

**Design principle:** analysts never query Silver directly for these answers.
Gold tables are small, fast, and pre-answered; the Streamlit dashboard reads
exclusively from them (via the exported CSVs).

**Run `00_setup` and `02_silver_transform` first.**

In [0]:
%run ./00_setup

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# READ SILVER AND FILTER TO CLEAN ROWS
# ─────────────────────────────────────────────────────────────────────────────
# Gold analyses clean rows only (dq_flag = False); flagged rows stay in Silver
# for audit. The Silver→Gold contract: clean rows are trusted as-is.

from pyspark.sql.functions import col, count, when, lit

df_clean = (
    spark.table(TABLE_SILVER)
    .filter(col("dq_flag") == False)
)

clean_count = df_clean.count()
print(f"Silver clean rows available for Gold: {clean_count:,}")
print(f"Table read from: {TABLE_SILVER}\n")
print("Ready to build Gold tables.")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# GOLD TABLE 1: significance_summary
# ─────────────────────────────────────────────────────────────────────────────
# Q: how many variants per clinical significance category?
# Grain: one row per category. Powers dashboard panel 1.
# pct uses clean_count as denominator → sums to ~100%.
# 
#
# DLT-ready structure — in a production workspace this would be declared as:
# @dlt.table(name="significance_summary",
#            comment="Variant counts by clinical significance category")
# def significance_summary():
#     return (spark.table("LIVE.variant_summary_clean")
#             .filter(col("dq_flag") == False)
#             .groupBy("ClinicalSignificance_clean")
#             .agg(count("*").alias("variant_count"))
#             .orderBy("variant_count", ascending=False))

from pyspark.sql.functions import count, round

df_significance = (
    df_clean
    .groupBy("ClinicalSignificance_clean")
    .agg(count("*").alias("variant_count"))
    .withColumn(
        "pct_of_total",
        round(col("variant_count") / clean_count * 100, 2)
    )
    .orderBy("variant_count", ascending=False)
)

# Drop existing table if present — schema may have changed
spark.sql(f"DROP TABLE IF EXISTS {TABLE_GOLD_SIGNIFICANCE}")

df_significance.write.format("delta").mode("overwrite").saveAsTable(TABLE_GOLD_SIGNIFICANCE)

print(f"Table written: {TABLE_GOLD_SIGNIFICANCE}")
print(f"Rows: {df_significance.count()}\n")
df_significance.show(truncate=False)

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# GOLD TABLE 2: pathogenic_by_gene
# ─────────────────────────────────────────────────────────────────────────────
# Q: which genes have the most actionable variants?
# Filter: Pathogenic + Likely pathogenic. Grain: one row per gene.
# Powers top-genes bar chart. (GeneSymbol uppercased in Silver → no split counts.)
#
# DLT-ready structure:
# @dlt.table(name="pathogenic_by_gene",
#            comment="Top genes by pathogenic variant count",
#            table_properties={"quality": "gold"})
# def pathogenic_by_gene():
#     return (spark.table("LIVE.variant_summary_clean")
#             .filter(col("dq_flag") == False)
#             .filter(col("ClinicalSignificance_clean")
#                     .isin(["Pathogenic", "Likely pathogenic"]))
#             .groupBy("GeneSymbol")
#             .agg(count("*").alias("pathogenic_variant_count"))
#             .orderBy("pathogenic_variant_count", ascending=False))

from pyspark.sql.functions import count

df_pathogenic = (
    df_clean
    .filter(col("ClinicalSignificance_clean").isin(PATHOGENIC_CATEGORIES))
    .groupBy("GeneSymbol")
    .agg(count("*").alias("pathogenic_variant_count"))
    .orderBy("pathogenic_variant_count", ascending=False)
)

spark.sql(f"DROP TABLE IF EXISTS {TABLE_GOLD_PATHOGENIC}")

df_pathogenic.write.format("delta").mode("overwrite").saveAsTable(TABLE_GOLD_PATHOGENIC)

print(f"Table written: {TABLE_GOLD_PATHOGENIC}")
print(f"Distinct genes with pathogenic variants: {df_pathogenic.count():,}\n")
print("Top 20 genes by pathogenic variant count:")
df_pathogenic.show(20, truncate=False)

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# GOLD TABLE 3: variants_by_chromosome
# ─────────────────────────────────────────────────────────────────────────────
# Q: variant distribution per chromosome + pathogenic proportion?
# count(when(...)) counts pathogenic in the same pass as count(*) for total.
# Grain: one row per chromosome. Powers chromosome chart.
#
# DLT-ready structure:
# @dlt.table(name="variants_by_chromosome",
#            comment="Variant counts and pathogenic proportion per chromosome",
#            table_properties={"quality": "gold"})
# def variants_by_chromosome():
#     ...

from pyspark.sql.functions import count, when, round, col

df_chromosome = (
    df_clean
    .groupBy("Chromosome")
    .agg(
        count("*").alias("total_variants"),
        count(
            when(col("ClinicalSignificance_clean")
                 .isin(PATHOGENIC_CATEGORIES), 1)
        ).alias("pathogenic_count")
    )
    .withColumn(
        "pathogenic_pct",
        round(col("pathogenic_count") / col("total_variants") * 100, 2)
    )
    .orderBy("total_variants", ascending=False)
)

spark.sql(f"DROP TABLE IF EXISTS {TABLE_GOLD_CHROMOSOME}")

df_chromosome.write.format("delta").mode("overwrite").saveAsTable(TABLE_GOLD_CHROMOSOME)

print(f"Table written: {TABLE_GOLD_CHROMOSOME}")
print(f"Rows: {df_chromosome.count()}\n")
df_chromosome.show(20, truncate=False)

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# GOLD TABLE 4: condition_burden
# ─────────────────────────────────────────────────────────────────────────────
# Q: which conditions carry the most actionable variants?
# PhenotypeList is pipe-separated → split + explode so each condition counts
# independently. Count = variant–condition links (multi-condition variants
# count once per condition). Placeholder conditions filtered out.
#
# DLT-ready structure:
# @dlt.table(name="condition_burden",
#            comment="Top conditions by pathogenic variant burden",
#            table_properties={"quality": "gold"})
# def condition_burden():
#     ...

from pyspark.sql.functions import count, split, explode, trim, col

df_condition = (
    df_clean
    .filter(col("ClinicalSignificance_clean").isin(PATHOGENIC_CATEGORIES))
    .filter(col("PhenotypeList").isNotNull())
    # PhenotypeList contains pipe-separated conditions — split and explode
    # explode turns one row with ["condition A", "condition B"]
    # into two rows, one per condition
    # .withColumn("condition", explode(split(col("PhenotypeList"), "\\|")))
    .withColumn("condition", explode(split(col("PhenotypeList"), "[|;]")))
    .withColumn("condition", trim(col("condition")))
    # Filter out generic placeholder conditions
    .filter(~col("condition").isin([
        "not provided",
        "not specified", 
        "See cases",
        ""
    ]))
    .groupBy("condition")
    .agg(count("*").alias("pathogenic_variant_count"))
    .orderBy("pathogenic_variant_count", ascending=False)
)

spark.sql(f"DROP TABLE IF EXISTS {TABLE_GOLD_CONDITION}")

df_condition.write.format("delta").mode("overwrite").saveAsTable(TABLE_GOLD_CONDITION)

print(f"Table written: {TABLE_GOLD_CONDITION}")
print(f"Distinct conditions: {df_condition.count():,}\n")
print("Top 20 conditions by pathogenic variant burden:")
df_condition.show(20, truncate=False)

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# GOLD VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
# Read all four tables back from Unity Catalog to confirm they persisted and
# are queryable. Two semantic spot checks: top gene = BRCA2, significance
# percentages sum to ~100%.

print("=" * 60)
print("GOLD VALIDATION")
print("=" * 60)

tables = {
    "significance_summary"    : TABLE_GOLD_SIGNIFICANCE,
    "pathogenic_by_gene"      : TABLE_GOLD_PATHOGENIC,
    "variants_by_chromosome"  : TABLE_GOLD_CHROMOSOME,
    "condition_burden"        : TABLE_GOLD_CONDITION,
}

all_ok = True
for name, table in tables.items():
    try:
        row_count = spark.table(table).count()
        print(f"  ✓ {name:30s}: {row_count:>7,} rows")
    except Exception as e:
        print(f"  ✗ {name:30s}: ERROR — {e}")
        all_ok = False

# Spot check — top pathogenic gene should be BRCA2
top_gene = (
    spark.table(TABLE_GOLD_PATHOGENIC)
    .orderBy("pathogenic_variant_count", ascending=False)
    .limit(1)
    .collect()[0]
)
print(f"\n  Spot check — top pathogenic gene:")
print(f"    {top_gene['GeneSymbol']} with {top_gene['pathogenic_variant_count']:,} variants")

# Spot check — significance percentages should sum to ~100
total_pct = (
    spark.table(TABLE_GOLD_SIGNIFICANCE)
    .agg({"pct_of_total": "sum"})
    .collect()[0][0]
)
print(f"\n  Spot check — significance pct sum: {total_pct:.2f}% (expect ~100%)")

print("\n" + "=" * 60)
if all_ok:
    print("  GOLD COMPLETE")
else:
    print("  GOLD INCOMPLETE — fix errors above before proceeding")
print("=" * 60)

In [0]:
for table, name in [
    ("workspace.clinvar.significance_summary", "gold_significance"),
    ("workspace.clinvar.pathogenic_by_gene", "gold_pathogenic"),
    ("workspace.clinvar.variants_by_chromosome", "gold_chromosome"),
    ("workspace.clinvar.condition_burden", "gold_condition"),
]:
    spark.table(table).toPandas().to_csv(
        f"/Volumes/workspace/clinvar/clean_tables/{name}.csv", index=False
    )
    print(f"Exported: {name}.csv")